In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
ETAS Network — Italy INGV Catalog (1985-2025)

Run as a script  : python ITALY_network_ETAS.py
Convert to notebook: python convert_to_notebook.py ITALY_network_ETAS.py notebooks/ITALY_network_ETAS.ipynb
"""

import logging
import pickle
import time
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import seaborn as sns

from src.network import build_etas_network
from src.metrics import estimate_gamma_mle, test_power_law, measure_pa_forest, plot_preferential_attachment
from src.assortativity import compute_assortativity, plot_assortativity, plot_knn, plot_rich_club
from src.centrality import (
    plot_centrality_correlation,
    plot_top_n_cells,
    plot_geo_top_n_interactive,
    plot_geo_centrality_overlap,
    compute_bb_fitness_events,
    plot_bb_fitness,
    plot_bb_fitness_theory,
    plot_bb_fitness_geo,
)
from src.community import (
    run_louvain,
    run_consensus_louvain,
    run_spectral,
    run_infomap,
    run_hdbscan_spectral,
    run_hdbscan_geo,
    run_bigclam,
    compare_partitions,
    plot_partition_scores,
    compute_nmi_matrix,
    plot_nmi_heatmap,
    plot_community_geo,
)
from src.viz import (
    analyze_degree_distribution,
    analyze_degree_distribution_log_binning,
    plot_ccdf_with_fit,
    plot_degree_distribution_linear,
)
from src.plotutils import setup_matplotlib, configure_saves, savefig, save_plotly

try:
    from IPython.display import display
except ImportError:
    display = print  # type: ignore[assignment]

logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")
sns.set_theme(style="whitegrid")
pio.renderers.default = "notebook"

DATA_DIR    = Path("data/INGV")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
(RESULTS_DIR / "data").mkdir(exist_ok=True)
(RESULTS_DIR / "cache").mkdir(exist_ok=True)
(RESULTS_DIR / "gephi").mkdir(exist_ok=True)

CUT_YEAR   = 1985
TARGET_CRS = "epsg:32632"  # UTM Zone 32N — Italy

# ETAS model parameters (Console et al. 2003, Italian seismicity)
ETAS_K          = 0.013    # productivity constant
ETAS_ALPHA      = 1.23     # magnitude scaling exponent
ETAS_C_DAYS     = 0.05     # Omori time offset (days)
ETAS_P          = 1.07     # Omori temporal decay exponent
ETAS_Q          = 1.5      # spatial decay exponent
ETAS_D_KM       = 1.5      # spatial scale factor (km)
ETAS_T_MAX_DAYS = 730.0    # look-back window (2 years)
ETAS_MU         = None     # None = auto-estimate (targets ~35% background events)

SAVE_PDF: bool = True
SAVE_JPG: bool = True

setup_matplotlib()
configure_saves(SAVE_JPG, SAVE_PDF, RESULTS_DIR / "figures" / "italy" / "etas")

## Data Loading

The Italy INGV catalog covers $M \geq 1.5$ events from 1985–2025.

The ETAS network uses **both spatial and temporal information** together with
magnitudes, like the BP/ZBZ models — but with a physically motivated probabilistic
kernel rather than the deterministic $\eta_{ij}$ metric.  Every event still links
to at most one parent, producing a directed forest.

Key distinction from ZBZ: ZBZ separates aftershocks from background using a
data-driven GMM threshold on the $\log_{10}\eta$ distribution.  ETAS uses a
physically derived space-time-magnitude kernel with literature parameters for Italy,
making background detection theoretically grounded in the Omori-Utsu and
Gutenberg-Richter laws.

**Expected output:** ~215,000 rows; build time ~10–20 min (same O(N×W) as BP/ZBZ).

In [ ]:
print("Loading Italy earthquake catalog...")
df = pd.read_csv(DATA_DIR / "italy_earthquakes_1985_2025.csv")
df["time"] = pd.to_datetime(df["time"], utc=True)
df_net = (
    df[df["time"].dt.year >= CUT_YEAR]
    .sort_values("time")
    .reset_index(drop=True)
)
print(f"Loaded {len(df_net):,} earthquakes "
      f"({df_net['time'].dt.year.min()}–{df_net['time'].dt.year.max()})")
print(f"RAM: {df_net.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
display(pd.concat([df_net.head(3), df_net.tail(3)]))

## ETAS Kernel Preview

Before building the network, we visualise the ETAS triggering kernel surface
to understand how parent–offspring linking probability depends on time and distance.

The triggering intensity from parent $i$ to future event $j$ is:

$$\kappa_{ij} = K \cdot e^{\alpha(m_i - m_{\min})} \cdot \frac{1}{(t_{ij} + c)^p} \cdot \frac{1}{(r_{ij}^2 + d^2)^q}$$

- **Temporal factor** $(t_{ij}+c)^{-p}$: Omori-Utsu law — aftershock rate decays as
a power law in time since the mainshock.  $c = 0.05$ days prevents the singularity
at $t=0$; $p = 1.07$ for Italy (Console et al. 2003).
- **Spatial factor** $(r_{ij}^2 + d^2)^{-q}$: modified power law in distance.
$d = 1.5$ km regularises the singularity at $r=0$; $q = 1.5$ gives faster spatial
decay than Omori's temporal decay — an aftershock cluster is more spatially
compact than it is temporally extended.
- **Magnitude factor** $e^{\alpha m_i}$: larger mainshocks produce exponentially
more offspring — consistent with the Gutenberg-Richter productivity relation.

Parameters from: Console, R., & Murru, M. (2001). *JGR*, 106(B5), 8699–8711.

**Expected output:** Two subplots showing how $\kappa$ decays with increasing
$t_{ij}$ and $r_{ij}$ for a reference mainshock $m_i = 5.0$.  The temporal
decay is steeper than the spatial decay at short distances — an aftershock
1 day later is much less probable than one 1 hour later, but an aftershock
10 km away is still quite probable.

In [ ]:
m_ref  = 5.0
m_min  = float(df_net["magnitude"].min())
t_grid = np.linspace(0.001, 30.0, 500)   # days
r_grid = np.linspace(0.1,   100.0, 500)  # km

kappa_t = (ETAS_K * np.exp(ETAS_ALPHA * (m_ref - m_min))
           / (t_grid + ETAS_C_DAYS)**ETAS_P
           / (0.0 + ETAS_D_KM**2)**ETAS_Q)  # at r=0

kappa_r = (ETAS_K * np.exp(ETAS_ALPHA * (m_ref - m_min))
           / (1.0 + ETAS_C_DAYS)**ETAS_P    # at t=1 day
           / (r_grid**2 + ETAS_D_KM**2)**ETAS_Q)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.semilogy(t_grid, kappa_t, color="steelblue")
ax1.set_xlabel("Time since mainshock $t_{ij}$ (days)")
ax1.set_ylabel("$\\kappa_{ij}$ (log scale)")
ax1.set_title(f"Temporal decay  ($m_i={m_ref}$, $r=0$)")

ax2.semilogy(r_grid, kappa_r, color="crimson")
ax2.set_xlabel("Distance $r_{ij}$ (km)")
ax2.set_ylabel("$\\kappa_{ij}$ (log scale)")
ax2.set_title(f"Spatial decay  ($m_i={m_ref}$, $t=1$ day)")

fig.suptitle(f"ETAS Kernel — K={ETAS_K}, α={ETAS_ALPHA}, c={ETAS_C_DAYS}d, p={ETAS_P}, q={ETAS_Q}, d={ETAS_D_KM}km",
             fontsize=10)
fig.tight_layout()
savefig("etas_kernel_preview_italy")
plt.show()

## Network Construction

Each earthquake $j$ is linked to its most probable ETAS parent $i^*$:

$$i^* = \arg\max_{i < j,\; t_{ij} < T_{\max}} \kappa_{ij}$$

An event is classified as **background** (no parent assigned) when
$\max_i \kappa_{ij} \leq \mu_{\text{threshold}}$.  The threshold $\mu$ is
auto-estimated as the 35th percentile of $\max_i \kappa_{ij}$ across all events,
targeting ~35% background fraction — consistent with stochastic declustering
estimates for Italian seismicity (Zhuang et al. 2002).

The result is a **directed forest** identical in structure to BP/ZBZ but derived
from the ETAS physical kernel.  Key differences from ZBZ:
- ZBZ threshold: data-driven (GMM trough on $\log_{10}\eta$ distribution)
- ETAS threshold: data-driven (percentile of $\max \kappa$ values)
- ZBZ metric: $\eta_{ij} = t_{ij}^\alpha r_{ij}^d 10^{-bm_i}$ (product form)
- ETAS kernel: $\kappa_{ij}$ (Omori temporal × power-law spatial × exponential magnitude)

*References:*
Ogata, Y. (1988). Statistical models for earthquake occurrences.
*JASA*, 83(401), 9–27.
Zhuang, J., Ogata, Y., & Vere-Jones, D. (2002). Stochastic declustering.
*JASA*, 97(458), 369–380.

**Expected output:** N=215k nodes; M≈140k–170k edges; background fraction ≈30–40%;
n_trees ≈30k–80k; build time ~10–20 min.  Compare background fraction to ZBZ's GMM
estimate — similar values confirm that both methods identify the same events as
background, validating the seismological interpretation.

In [ ]:
print("Building ETAS network (this takes ~10–20 min for 215k events)…")
t_build = time.time()
G = build_etas_network(
    df_net,
    K=ETAS_K,
    alpha_etas=ETAS_ALPHA,
    c_days=ETAS_C_DAYS,
    p=ETAS_P,
    q=ETAS_Q,
    d_km=ETAS_D_KM,
    t_max_days=ETAS_T_MAX_DAYS,
    mu_threshold=ETAS_MU,
    target_crs=TARGET_CRS,
)
print(f"Build time: {time.time() - t_build:.1f}s")
print(f"Nodes: {G.number_of_nodes():,}  Edges: {G.number_of_edges():,}")

n_roots = sum(1 for _, d in G.in_degree() if d == 0)
n_trees = nx.number_weakly_connected_components(G)
bg_frac = 100.0 * n_roots / G.number_of_nodes()
print(f"Roots (background): {n_roots:,} ({bg_frac:.1f}%)")
print(f"Trees (WCC): {n_trees:,}")

## Tree Structure Verification

A valid ETAS directed forest must satisfy:
1. **Max in-degree ≤ 1** — each event has at most one ETAS parent
2. **Background events** — roots (in-degree 0) are the events assigned no parent
3. **WCC sizes** — the distribution of aftershock tree sizes reflects the
branching structure of the ETAS process

The ETAS branching ratio $n = K \int e^{\alpha m} dF(m)$ where $F(m)$ is the
magnitude distribution controls the average offspring count.  For a sub-critical
process ($n < 1$, required for stationarity), the tree size distribution follows
a power law $P(s) \sim s^{-3/2}$ (Borel distribution for Galton-Watson trees).

Comparing tree-size distributions between ETAS and ZBZ reveals whether the
GMM-based and kernel-based declustering methods agree on which events form the
same aftershock sequences.

**Expected output:** max in-degree = 1; WCC size distribution spanning 1 (isolated
events) to tens of thousands (the aftershock sequence of the largest mainshocks);
background fraction 20–40%.

In [ ]:
max_indeg = max(d for _, d in G.in_degree())
print(f"Max in-degree: {max_indeg}  (must be ≤ 1)")
assert max_indeg <= 1, "ETAS forest invariant violated!"

wcc_sizes = sorted([len(c) for c in nx.weakly_connected_components(G)], reverse=True)
print(f"Largest 5 trees: {wcc_sizes[:5]}")
print(f"Trees of size 1 (isolated): {wcc_sizes.count(1):,}")
print(f"Trees of size > 10: {sum(1 for s in wcc_sizes if s > 10):,}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(wcc_sizes, bins=100, log=True, color="steelblue", alpha=0.7)
ax.set_xlabel("Tree size (events)")
ax.set_ylabel("Count (log scale)")
ax.set_title("ETAS Aftershock Tree Size Distribution — Italy")
fig.tight_layout()
savefig("etas_tree_sizes_italy")
plt.show()

## Hub Map — 2D

High out-degree events in the ETAS network are the most productive mainshocks —
events that triggered many direct offspring in the ETAS model.

ETAS hubs differ from BP hubs in that the ETAS kernel uses **absolute distance**
(km via projected coordinates) and a physically calibrated spatial scale $d=1.5$ km,
while BP uses haversine + fractal dimension $d_f=1.6$.  For moderate-magnitude events
at short inter-event distances, the two kernels assign similar parents; for
background-rate events the ETAS background separation gives more background roots
than BP, making the ETAS hubs those events that dominate their local space-time
neighbourhood beyond the background threshold.

**Expected output:** Map dominated by the same mainshocks as ZBZ (L'Aquila 2009,
Amatrice 2016) but with potentially different productivity rankings depending on
how the ETAS spatial kernel weights distance vs the BP metric.

In [ ]:
out_degrees = dict(G.out_degree())
df_nodes = pd.DataFrame([
    {
        "event_idx": n,
        "out_degree": out_degrees[n],
        "lat":        G.nodes[n]["lat"],
        "lon":        G.nodes[n]["lon"],
        "magnitude":  G.nodes[n]["magnitude"],
    }
    for n in G.nodes()
])
df_hubs = df_nodes.nlargest(100, "out_degree").copy()
print(f"Top 100 hubs: out-degree range [{df_hubs['out_degree'].min():.0f}, {df_hubs['out_degree'].max():.0f}]")

df_hubs = df_hubs.sort_values("out_degree")
_odmin  = df_hubs["out_degree"].min()
_odmax  = df_hubs["out_degree"].max()
df_hubs["marker_size"] = 8 + 27 * (df_hubs["out_degree"] - _odmin) / max(_odmax - _odmin, 1)

_IT_BOUNDS = dict(west=3, east=22, south=34, north=48)
MAP_WIDTH  = 770
MAP_HEIGHT = 700

fig = px.scatter_map(
    df_hubs, lat="lat", lon="lon",
    color="out_degree", size="marker_size",
    size_max=35, opacity=0.85,
    color_continuous_scale="plasma",
    map_style="carto-positron",
    hover_name="event_idx",
    hover_data={"magnitude": ":.2f", "out_degree": True, "marker_size": False},
    title="ETAS Network Hubs: Top 100 Most Productive Events — Italy",
)
fig.update_layout(
    margin={"r": 0, "t": 40, "l": 0, "b": 0},
    width=MAP_WIDTH, height=MAP_HEIGHT,
    map=dict(center={"lat": 41.9, "lon": 12.5}, zoom=0, bounds=_IT_BOUNDS),
)
save_plotly(fig, "etas_hub_map_2d_italy")
fig.show()

## Out-Degree Distribution — Linear

Linear-scale histogram of out-degrees (number of direct ETAS offspring per event).
The ETAS branching process predicts a geometric offspring distribution:

$$P(\text{out-degree} = k) \approx (1-n) \cdot n^k$$

where $n = K \cdot \langle e^{\alpha m} \rangle$ is the branching ratio.
For Italy with $n \approx 0.3$–$0.7$ (sub-critical), the distribution is steeply
decaying with most events having 0–2 direct offspring.

**Expected output:** Strongly zero-inflated distribution — the majority of events
are leaves (out-degree 0, no ETAS offspring) or background events.  A long tail
(out-degree > 100) for the largest mainshocks.

In [ ]:
plot_degree_distribution_linear(G, "ETAS Italy (out-degree)", max_degree=50, weighted=False)

## Out-Degree Distribution — Log-Log

Log-log scatter of $P(k_{\text{out}})$ vs $k_{\text{out}}$.  The ETAS branching
process generates a power-law offspring distribution in the critical limit ($n \to 1$).
For sub-critical Italy ($n < 1$), the tail decays faster than a power law, but may
appear approximately linear over a limited range.

A log-log plot that falls below the NVG/HVG curves confirms ETAS produces sparser
connectivity: background events have out-degree 0, compressing the degree sequence
toward very low values compared to the visibility graphs.

In [ ]:
analyze_degree_distribution(G, "ETAS Italy (out-degree)")

## Out-Degree Distribution — Log Binning

Log-binned log-log stabilises the sparse tail for MLE estimation.

In [ ]:
analyze_degree_distribution_log_binning(G, "ETAS Italy (out-degree)", k_min_fit=2)

## Out-Degree Distribution — CCDF

CCDF $P(K_{\text{out}} \geq k)$ for the out-degree sequence, avoiding binning.
Compare to the ZBZ CCDF: similar tail slope would confirm that both declustering
methods produce aftershock sequences of the same statistical character.

In [ ]:
plot_ccdf_with_fit(G, "ETAS Italy (out-degree)", k_min_fit=2)

## MLE Gamma — Branching Exponent

Maximum-likelihood estimate of the out-degree tail exponent (Clauset et al. 2009):

$$\hat{\gamma} = 1 + n_{\text{tail}}\left[\sum_{i}\ln\frac{k_i}{k_{\min}}\right]^{-1}$$

In the ETAS branching process, the out-degree distribution follows a power law
$P(k) \sim k^{-\gamma}$ for large $k$, with exponent determined by the branching
ratio $n$ and the magnitude distribution.  For Italian parameters ($\alpha=1.23$,
$b=1.0$), the theoretical ETAS offspring exponent is $\gamma = 1 + b/\alpha \approx 1.81$
(Helmstetter & Sornette 2002).

*Reference:* Helmstetter, A., & Sornette, D. (2002). Subcritical and supercritical
regimes in epidemic models of earthquake aftershocks. *JGR*, 107(B10), 2237.

**Expected output:** $\hat{\gamma} \approx 1.5$–$2.5$; this is typically lower (heavier
tail) than BP because the ETAS spatial kernel clusters more offspring near large events
and the background separation leaves more "unlinked" events as roots, concentrating
the offspring count on true parents.

In [ ]:
all_out = [d for _, d in G.out_degree()]
gamma_etas = estimate_gamma_mle(all_out, k_min=2)
print(f"MLE γ (out-degree, k_min=2): {gamma_etas:.3f}")
print(f"  Theoretical ETAS exponent: γ = 1 + b/α = 1 + 1.0/{ETAS_ALPHA:.2f} = {1 + 1.0/ETAS_ALPHA:.3f}")

result_etas = test_power_law(all_out, k_min=2)
print(f"  CSN test: γ={result_etas['gamma']} (σ={result_etas['sigma']})  "
      f"R={result_etas['R']}  p={result_etas['p_value']}  → {result_etas['verdict']}")

## Preferential Attachment

In the ETAS forest each aftershock $j$ is assigned to the parent $i$ that
maximises the ETAS triggering kernel $\kappa_{ij}$.  The attachment kernel
$\pi(k_{\text{out}}) \propto k_{\text{out}}^{\alpha}$ is measured by replaying
the chronological edge sequence and recording, for each parent–child assignment,
the out-degree of the parent at the moment of attachment:

$$\pi(k_{\text{out}}) = \frac{\sum_{i:\,k_i^{\text{out}}(t)=k} \Delta k_i}
{\#\{i:\,k_i^{\text{out}}(t)=k\}}$$

Unlike BP and ZBZ (which use the same $\eta$ metric), the ETAS kernel
explicitly decouples magnitude ($10^{\alpha_E m_i}$) from time-decay
($(t_{ij}+c)^{-p}$) and space-decay ($(r_{ij}^2+d^2)^{-q}$).
Comparing $\alpha_{ETAS}$ with $\alpha_{BP}$ and $\alpha_{ZBZ}$ tests
whether the choice of parent-assignment rule changes the degree-growth
mechanism: all three should yield $\alpha \approx 1$ if the underlying
seismicity is BA-like, and $\alpha < 1$ if magnitude-dependent parent selection
creates a winner-take-all distortion.

*Reference:* Jeong H., Néda Z. & Barabási A.-L. (2003). Measuring preferential
attachment in evolving networks. *Europhysics Letters*, 61(4), 567–572.

In [ ]:
print("Measuring preferential attachment kernel (ETAS forest)…")
ks_pa, pi_k_pa, alpha_pa = measure_pa_forest(G, df_net)
plot_preferential_attachment(ks_pa, pi_k_pa, alpha_pa, title="ETAS Italy (forest PA)")

## Macroscopic Metrics

The ETAS forest has the same structural properties as ZBZ (directed forest with many
roots) but potentially different tree sizes and depth distributions because the ETAS
kernel weights time and space differently from the $\eta$ metric.

Key comparison with ZBZ:
- **Background fraction:** ETAS auto-targets 35% (ZBZ is GMM-derived, ~20–40%).
Similar values validate that both methods identify the same background events.
- **Max tree depth:** ETAS spatial decay is faster than BP ($q=1.5$ vs fractal $d=1.6$),
potentially producing shallower trees with more branching near the mainshock.
- **Average path length** in the GCC: shorter than BP (which forms one spanning tree)
but longer than ZBZ if ETAS is less aggressive at cutting long chains.

**Expected output:** $n_{\text{trees}} \approx$ 30k–80k; largest tree 10k–50k events
(the L'Aquila or Amatrice aftershock sequence); background fraction ~30–40%;
max tree depth ~50–500 hops.

In [ ]:
N_nodes = G.number_of_nodes()
M_edges = G.number_of_edges()
print(f"Nodes: {N_nodes:,}  Edges: {M_edges:,}")
print(f"Background (roots): {n_roots:,} ({bg_frac:.1f}%)")
print(f"Trees: {n_trees:,}")

# Largest tree metrics
gcc = max(nx.weakly_connected_components(G), key=len)
print(f"Largest tree: {len(gcc):,} nodes")

# Approx avg path length via BFS on GCC
G_gcc = G.subgraph(gcc)
print("Approximating avg path length in largest tree (500 BFS seeds)…")
rng      = np.random.default_rng(42)
seeds_apl = list(rng.choice(list(G_gcc.nodes()), size=min(500, len(gcc)), replace=False))
apl_vals  = []
for s in seeds_apl:
    lengths = nx.single_source_shortest_path_length(G_gcc, s)
    apl_vals.extend(lengths.values())
avg_L = np.mean(apl_vals)
print(f"Approx avg path length (largest tree): {avg_L:.2f} hops")

# Max depth from roots
roots = [n for n, d in G.in_degree() if d == 0]
sample_roots = list(rng.choice(roots, size=min(200, len(roots)), replace=False))
max_depth = 0
for r in sample_roots:
    lengths = nx.single_source_shortest_path_length(G, r)
    if lengths:
        max_depth = max(max_depth, max(lengths.values()))
print(f"Max tree depth (sampled from {len(sample_roots)} roots): {max_depth}")

## Centrality Analysis

## Centrality Analysis — 12 Measures

Twelve centrality measures identify events most influential under the ETAS model:

### Degree family
- **In-Degree**: 0 for background events (ETAS background process); 1 for all
triggered offspring.  Distinguishes ETAS-background from ETAS-triggered events —
a finer distinction than ZBZ because background probability is continuous.
- **Out-Degree**: direct ETAS offspring count — events with high ETAS kernel
amplitude (large $K_0 e^{\alpha M}$, small $c$) accumulate high out-degree.
Higher than ZBZ out-degree for the same event indicates the ETAS kernel assigns
a larger spatial/temporal footprint than the ZBZ η metric.
- **Degree**: total connectivity.

### Path-based measures
- **PageRank**: flows from roots toward leaves in the directed forest.
High-PR nodes accumulated rank through a chain of productive ancestors;
may differ substantially from ZBZ-PR because ETAS parent assignment changes
tree topology.
- **Harmonic**: handles the disconnected multi-tree forest (closeness = 0 across
weakly connected components); essential for ETAS where the background process
creates many isolated trees.
- **Betweenness** ($k = 500$): "relay" mainshocks that connect generations of
aftershocks — events on the unique path between roots and distant leaves.

### Spectral
- **Eigenvector / Katz**: zero-filled if forest is disconnected; Katz
($\alpha = 0.85 / k^{\mathrm{out}}_{\max}$) is more robust for sparse forests.
- **HITS Hub / Authority**: Hub ≈ out-degree; Authority ≈ attracted by productive hubs.

### Local topology
- **Clustering / Triangles**: expected ≈ 0 for a pure forest; non-zero values
indicate rare events assigned as offspring to two different parents by the
probabilistic ETAS sampling.

In [ ]:
print("Computing centralities for ETAS network…")
_t0_cent = time.time()

G_und_all = G.to_undirected()
G_und_all.remove_edges_from(nx.selfloop_edges(G_und_all))
_N = G.number_of_nodes()

in_deg_cent  = nx.in_degree_centrality(G)
out_deg_cent = nx.out_degree_centrality(G)
deg_cent     = {n: G.out_degree(n) / max(_N - 1, 1) for n in G.nodes()}
print(f"  Degree (in/out/total) done ({time.time()-_t0_cent:.1f}s)")

harm_cent = nx.harmonic_centrality(G)
print(f"  Harmonic done ({time.time()-_t0_cent:.1f}s)")

pr_cent = nx.pagerank(G, weight="weight")
print(f"  PageRank done ({time.time()-_t0_cent:.1f}s)")

bet_cent = nx.betweenness_centrality(G, k=min(500, _N), seed=42, normalized=True)
print(f"  Betweenness done ({time.time()-_t0_cent:.1f}s)")

try:
    eig_cent = nx.eigenvector_centrality(G_und_all, weight="weight", max_iter=500, tol=1e-6)
except nx.PowerIterationFailedConvergence:
    try:
        eig_cent = nx.eigenvector_centrality_numpy(G_und_all, weight="weight")
    except nx.AmbiguousSolution:
        eig_cent = {n: 0.0 for n in G.nodes()}
        print("  Eigenvector: skipped (forest is disconnected)")
print(f"  Eigenvector done ({time.time()-_t0_cent:.1f}s)")

_max_out  = max(G.out_degree(n) for n in G.nodes()) or 1
_alpha_katz = 0.85 / _max_out
try:
    katz_cent = nx.katz_centrality(
        G, alpha=_alpha_katz, weight="weight", normalized=True, max_iter=1000, tol=1e-6)
except nx.PowerIterationFailedConvergence:
    katz_cent = nx.katz_centrality_numpy(G, alpha=_alpha_katz, weight="weight")
print(f"  Katz done ({time.time()-_t0_cent:.1f}s)")

try:
    hits_hub, hits_auth = nx.hits(G_und_all, max_iter=1000, tol=1e-6)
except nx.PowerIterationFailedConvergence:
    _zeros    = {n: 0.0 for n in G.nodes()}
    hits_hub  = _zeros.copy()
    hits_auth = _zeros.copy()
print(f"  HITS done ({time.time()-_t0_cent:.1f}s)")

clust_cent = nx.clustering(G_und_all, weight="weight")
print(f"  Clustering done ({time.time()-_t0_cent:.1f}s)")
tri_count = nx.triangles(G_und_all)
print(f"  Triangles done ({time.time()-_t0_cent:.1f}s total)")

df_centrality = pd.DataFrame([
    {
        "cell_id":     n,
        "lat":         G.nodes[n]["lat"],
        "lon":         G.nodes[n]["lon"],
        "depth_km":    G.nodes[n]["depth_km"],
        "In_Degree":   in_deg_cent.get(n, 0.0),
        "Out_Degree":  out_deg_cent.get(n, 0.0),
        "Degree":      deg_cent.get(n, 0.0),
        "PageRank":    pr_cent.get(n, 0.0),
        "Harmonic":    harm_cent.get(n, 0.0),
        "Betweenness": bet_cent.get(n, 0.0),
        "Eigenvector": eig_cent.get(n, 0.0),
        "Katz":        katz_cent.get(n, 0.0),
        "HITS_Hub":    hits_hub.get(n, 0.0),
        "HITS_Auth":   hits_auth.get(n, 0.0),
        "Clustering":  clust_cent.get(n, 0.0),
        "Triangles":   float(tri_count.get(n, 0)),
    }
    for n in G.nodes()
    if "lat" in G.nodes[n] and "lon" in G.nodes[n]
])

for metric, label in [
    ("In_Degree",   "Background Events (In-Degree = 0)"),
    ("Out_Degree",  "ETAS Productivity (Out-Degree)"),
    ("Degree",      "Most Productive Events"),
    ("PageRank",    "Aftershock Chain Recipients"),
    ("Harmonic",    "Topological Reach (Harmonic)"),
    ("Betweenness", "Relay Mainshocks"),
    ("HITS_Hub",    "ETAS Hubs"),
    ("HITS_Auth",   "ETAS Authorities"),
    ("Clustering",  "Multi-Path Events (Clustering)"),
    ("Triangles",   "Feedback Loops (Triangles)"),
]:
    print(f"\n--- Top 5: {label} ({metric}) ---")
    display(df_centrality.nlargest(5, metric)[
        ["cell_id", "lat", "lon", "depth_km", metric]])

plot_centrality_correlation(df_centrality, "ETAS Italy")
plot_top_n_cells(df_centrality, top_n=10, title="ETAS Italy")

## Centrality Geo Maps

Geographic projection of the top-10 events per centrality metric.

ETAS centrality maps should largely agree with ZBZ — both identify the same
major mainshocks as hubs.  Differences in top-10 composition reveal where the
physically calibrated ETAS kernel and the data-driven ZBZ GMM disagree on
parent assignment:
- An event that is a top-10 hub in ZBZ but not ETAS was likely classified as
background by ETAS (its κ_max < μ_threshold) despite being below the ZBZ η₀.
- An event that is ETAS hub but not ZBZ hub was assigned more ETAS offspring due
to its stronger spatial kernel footprint at the Italy-calibrated parameters.

**Expected output:** Similar geographic pattern to ZBZ, concentrated along the
Apennines.  The overlap map likely shows 3–6 events with multi-metric dominance —
the seismically most influential events in 40 years of Italian seismicity under
both declustering paradigms.

In [ ]:
plot_geo_top_n_interactive(
    df_centrality, top_n=10,
    title="ETAS Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
)
plot_geo_centrality_overlap(
    df_centrality, top_n=10,
    title="ETAS Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
)

## Bianconi-Barabasi Fitness

The ETAS model uses magnitude-dependent triggering ($10^{\alpha_E m_i}$ weight),
which means larger events are systematically more likely to be selected as
parents.  The fitness estimator $\hat{\beta}_i = \ln k_i(T) / \ln(T/t_i)$
asks, on top of this magnitude-driven mechanism, whether some events are
intrinsically more productive than their magnitude and timing predict:

$$\hat{\beta}_i = \frac{\ln k_i(T)}{\ln(T/t_i)}$$

In the ETAS forest, high $\hat{\beta}$ identifies events that triggered many
aftershocks rapidly — a Bose-Einstein condensation signature would indicate
that one or a few events (e.g. Amatrice 2016 M 6.2, L'Aquila 2009 M 6.3)
captured a macroscopic fraction of all parent-child links in the catalog.

Comparing $\rho(\hat{\beta})_{ETAS}$ against $\rho(\hat{\beta})_{BP}$ and
$\rho(\hat{\beta})_{ZBZ}$ tests whether the choice of parent-assignment rule
changes which events are identified as the network's dominant fitness outliers.

Three theoretical regimes: equal-fitness BA, uniform fitness, condensation.

*References:* Bianconi G. & Barabási A.-L. (2001). *EPL* 54, 436–442;
*PRL* 86, 5632–5635.

In [ ]:
print("Computing Bianconi-Barabasi fitness (ETAS events)…")
df_bb = compute_bb_fitness_events(G, df_net)
print(f"  {len(df_bb)} events with valid β̂; β range [{df_bb['fitness_beta'].min():.3f}, {df_bb['fitness_beta'].max():.3f}]")
plot_bb_fitness(df_bb, title="ETAS Italy")
plot_bb_fitness_theory(df_bb, gamma=gamma_etas, title="ETAS Italy")
plot_bb_fitness_geo(
    df_bb,
    title="ETAS Italy",
    center_lat=41.9, center_lon=12.5, zoom=0,
    bounds=_IT_BOUNDS, height=MAP_HEIGHT, width=MAP_WIDTH,
)

## Community Detection — Full Suite

Seven community-detection algorithms are applied to the undirected ETAS forest
(self-loops removed before conversion).  Modularity

$$Q = \frac{1}{2m}\sum_{ij}\left[A_{ij} - \frac{k_i k_j}{2m}\right]\delta(c_i, c_j)$$

(Newman & Girvan 2004) is the primary optimisation target for graph-based methods,
but density- and affiliation-based methods operate on complementary representations.

ETAS communities identify spatially and temporally coherent aftershock clusters —
groups of events that are more interconnected within the ETAS parent-offspring graph
than between groups.  They are expected to be more **seismologically meaningful**
than BP communities because the ETAS background separation removes long-range
"spanning" edges that connect distant events only because no closer candidate
existed — edges that inflate tree depth in BP without reflecting genuine aftershock
triggering.  Comparing ETAS vs ZBZ community structure (NMI) directly answers
whether the two declustering paradigms (ETAS kernel vs GMM threshold) identify the
same aftershock clusters.

**Graph-based methods**
- **Louvain** (Blondel et al. 2008): greedy modularity maximisation on the
undirected forest; fast and scalable.
- **Consensus Louvain** (Lancichinetti & Fortunato 2012): 100-run ensemble average
over co-assignment frequencies; the most reproducible partition.
- **Spectral** (Von Luxburg 2007): restricted to the giant component to avoid a
rank-deficient Laplacian from the disconnected multi-tree ETAS forest.
- **InfoMap** (directed; Rosvall & Bergstrom 2008): run on the directed ETAS forest,
respecting the probabilistic parent→offspring triggering direction.

**Density-based methods**
- **HDBSCAN-Spectral** (Campello et al. 2013): applies HDBSCAN to the
$k$-dimensional spectral embedding of the normalised graph Laplacian of the giant
component.  No $k$ pre-specification is required; noise points are excluded.
The spectral embedding captures the probabilistic aftershock-chain structure of
the ETAS forest independently of any modularity assumption.
- **HDBSCAN-Geographic**: same HDBSCAN algorithm applied to projected $(x, y)$
node coordinates in kilometres (EPSG:32632); no graph structure is used.
Comparing with HDBSCAN-Spectral tests whether the probabilistic ETAS communities
carry additional temporal and triggering information beyond what simple spatial
proximity would predict.

**Overlapping-community method**
- **BigCLAM** (Yang & Leskovec 2013): solves for an $N \times k$ affiliation
matrix $F$ via non-negative matrix factorisation; the hard partition is recovered
by $\arg\max_c F_{ic}$.  In the ETAS context, events with high background
probability that fall at the periphery of two distinct triggered clusters may
carry genuine dual affiliation — a physically meaningful overlap not captured by
the hard-partition methods.

**Partition quality scoring**
All seven partitions are evaluated on 9 quality metrics — modularity, conductance,
coverage, Ncut, map equation, DC-SBM log-likelihood, Surprise, geographic
compactness, and depth coherence — via `compare_partitions`.

*References:* Newman & Girvan (2004) *Phys. Rev. E* 69, 026113;
Blondel et al. (2008) *J. Stat. Mech.* P10008;
Rosvall & Bergstrom (2008) *PNAS* 105, 1118;
Campello et al. (2013) *ECML-PKDD* 160–175;
Yang & Leskovec (2013) *ACM TKDD* 7(3), 1–42.

**Expected output:** 10–50 communities; NMI between ETAS and ZBZ Louvain partitions
$> 0.5$ would confirm broad agreement between the two declustering paradigms;
$< 0.3$ would indicate that the ETAS kernel and the GMM threshold identify
structurally different aftershock clusters.  HDBSCAN-Geographic NMI with
HDBSCAN-Spectral quantifies how much community structure is geography-driven
versus encoded in the ETAS probabilistic triggering topology.

In [ ]:
print("Running community detection on ETAS forest…")

G_und_comm = G.to_undirected()
G_und_comm.remove_edges_from(nx.selfloop_edges(G_und_comm))

print("Louvain…")
community_map = run_louvain(G_und_comm, seed=42)
k_louvain     = len(set(community_map.values()))
print(f"  {k_louvain} communities")
plot_community_geo(
    G_und_comm, community_map,
    title="ETAS Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=50, method_name="Louvain",
)

print("Consensus Louvain (100 runs)…")
community_map_consensus = run_consensus_louvain(G_und_comm, n_runs=100, seed=42)
print(f"  {len(set(community_map_consensus.values()))} communities")
plot_community_geo(
    G_und_comm, community_map_consensus,
    title="ETAS Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=50, method_name="Consensus Louvain",
)

print(f"Spectral (k={min(k_louvain, 8)})…")
_k_spec = min(k_louvain, 8)
community_map_spectral = run_spectral(G_und_comm, k=_k_spec, seed=42)
print(f"  {len(set(community_map_spectral.values()))} communities")
plot_community_geo(
    G_und_comm, community_map_spectral,
    title="ETAS Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=50, method_name="Spectral",
)

print("InfoMap (directed)…")
community_map_infomap = run_infomap(G, directed=True, seed=42)
print(f"  {len(set(community_map_infomap.values()))} communities")
plot_community_geo(
    G_und_comm, community_map_infomap,
    title="ETAS Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=50, method_name="InfoMap",
)

partitions = {
    "Louvain":   community_map,
    "Consensus": community_map_consensus,
    "Spectral":  community_map_spectral,
    "InfoMap":   community_map_infomap,
}
print("Computing NMI…")
nmi_df = compute_nmi_matrix(partitions)
display(nmi_df.round(3))
plot_nmi_heatmap(nmi_df, title="ETAS Italy")

print("HDBSCAN-Spectral…")
community_map_hdbscan_spec = run_hdbscan_spectral(G_und_comm, min_cluster_size=10, seed=42)
print(f"  {len(set(community_map_hdbscan_spec.values()))} clusters")
plot_community_geo(
    G_und_comm, community_map_hdbscan_spec,
    title="ETAS Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=20, method_name="HDBSCAN-Spectral",
)

print("HDBSCAN-Geographic…")
community_map_hdbscan_geo = run_hdbscan_geo(G_und_comm, min_cluster_size=10)
print(f"  {len(set(community_map_hdbscan_geo.values()))} clusters")
plot_community_geo(
    G_und_comm, community_map_hdbscan_geo,
    title="ETAS Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=20, method_name="HDBSCAN-Geographic",
)

print("BigCLAM…")
F_bigclam, community_map_bigclam = run_bigclam(
    G_und_comm, k=k_louvain, n_iter=100, lr=0.005, seed=42,
)
print(f"  {len(set(community_map_bigclam.values()))} non-empty communities")
plot_community_geo(
    G_und_comm, community_map_bigclam,
    title="ETAS Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=20, method_name="BigCLAM",
)

_common_ext = (set(community_map_hdbscan_spec) & set(community_map_hdbscan_geo)
               & set(community_map_bigclam) & set(community_map))
partitions_ext = {
    **{k: {n: v[n] for n in _common_ext if n in v} for k, v in partitions.items()},
    "HDBSCAN-Spec": {n: community_map_hdbscan_spec[n] for n in _common_ext},
    "HDBSCAN-Geo":  {n: community_map_hdbscan_geo[n]  for n in _common_ext},
    "BigCLAM":      {n: community_map_bigclam[n]       for n in _common_ext},
}
nmi_ext = compute_nmi_matrix(partitions_ext)
display(nmi_ext.round(3))
plot_nmi_heatmap(nmi_ext, title="ETAS Italy — all methods")

scores_df = compare_partitions(G_und_comm, partitions_ext, cell_size_km=10.0)
display(scores_df.round(4))
plot_partition_scores(scores_df, title="ETAS Italy")

## Community Markov Flow

The Louvain partition coarse-grains the directed ETAS forest into a $K \times K$
flow matrix $F_{ij}$ counting directed edges from community $i$ to community $j$.
Row-normalisation gives the Markov transition matrix:

$$T_{ij} = \frac{F_{ij}}{\sum_j F_{ij}}$$

In the ETAS context, $T_{ij}$ is the probability that an ETAS offspring event falls
in community $j$ given that its parent is in community $i$.  **High self-retention**
($T_{ii} \approx 1$) indicates that ETAS communities are self-contained aftershock
clusters — offspring stay within the same seismic source region as their parents.
**High off-diagonal entries** identify pairs of communities with frequent cross-triggering —
geographically adjacent fault segments that regularly trigger each other.

*Reference:* Rosvall, M., & Bergstrom, C. T. (2008). *PNAS*, 105(4), 1118–1123.

**Expected output:** Higher self-retention than BP (which has one tree with many
cross-community edges) because ETAS cuts background connections.  Self-retention
$T_{ii} \approx 0.85$–$0.99$; compare directly to ZBZ values as a validation of
declustering consistency.

In [ ]:
from src.community_flow import (
    build_community_flow_matrix,
    compute_markov_chain,
    compute_stationary_distribution,
    community_flow_stats,
    plot_flow_heatmap,
    plot_flow_entropy,
    plot_stationary_distribution,
)

print("Building community flow matrix (Louvain partition, directed ETAS forest)…")
flow_count_df = build_community_flow_matrix(G, community_map)
T_markov  = compute_markov_chain(flow_count_df)
stat_dist = compute_stationary_distribution(T_markov)
df_flow   = community_flow_stats(T_markov, flow_count_df, stat_dist)

K_comm = T_markov.shape[0]
print(f"Markov chain: {K_comm} community states")
print(f"Mean self-retention:  {df_flow['self_retention'].mean():.3f}")
print(f"Mean outflow entropy: {df_flow['entropy'].mean():.3f} bits "
      f"(max = log₂({K_comm}) = {np.log2(K_comm):.3f})")
print(f"Dominant attractor:   Community {df_flow.iloc[0]['community']} "
      f"(π = {df_flow.iloc[0]['stationary']:.4f})")
display(df_flow)

plot_flow_heatmap(T_markov, flow_count_df, title="ETAS Italy")
plot_flow_entropy(df_flow, K=K_comm, title="ETAS Italy")
plot_stationary_distribution(df_flow, title="ETAS Italy")

out_path = RESULTS_DIR / "data" / "italy_etas_community_flow.csv"
df_flow.to_csv(out_path, index=False)
print(f"Community flow stats saved → {out_path}")

## Assortativity

Newman (2003) assortativity tests whether ETAS parent–offspring links preferentially
connect events with similar magnitude, depth, or degree.

**Magnitude assortativity** $r$: The ETAS kernel weights large-magnitude parents
exponentially ($e^{\alpha m}$), so large events tend to be parents of other
moderate-to-large events (their kernel footprint reaches further).  This should
produce slight **positive** magnitude assortativity — large events link to
relatively larger offspring compared to BP, where the metric also weights distance.

**Degree assortativity** (out-degree): In the ETAS forest, high-productivity
mainshocks are connected to many low-productivity aftershocks — expected to be
**negatively** assortative (disassortative), like BP and ZBZ.

Comparing ETAS vs ZBZ assortativity directly tests whether the two declustering
methods produce similar parent–offspring magnitude relationships.

*Reference:* Newman, M. E. J. (2003). *Physical Review E*, 67(2), 026126.

**Expected output:** Magnitude $r \approx -0.10$ to $+0.10$ (mild, similar to ZBZ);
degree $r \approx -0.3$ to $-0.5$ (disassortative hub-spoke structure).

In [ ]:
print("Computing assortativity…")
df_assort = compute_assortativity(G)
display(df_assort)
plot_assortativity(G, title="ETAS Italy")

print("Average nearest-neighbour degree k_nn(k):")
plot_knn(G, title="ETAS Italy", gamma=gamma_etas)

print("Rich-club coefficient:")
plot_rich_club(G, title="ETAS Italy")

## Export Results

Centrality metrics, community assignments, and the network are persisted for
downstream comparison with the other six Italy network models.

**CSV** (`italy_etas_network_metrics.csv`): one row per node; centrality scores +
Louvain community label.  Compare ETAS productivity rankings (out-degree) against
ZBZ productivity rankings — agreement validates both declustering methods.

**Pickle** (`italy_etas.pkl`): full `nx.DiGraph` with node attributes including
`is_background` (bool) flag for each event.  Load to quickly access the ETAS
parent–offspring structure without re-running the 10–20 min build.

**GEXF** (`italy_etas.gexf`): Gephi export.  Colour by `community` or `magnitude`.
Use hierarchical layout (Sugiyama) to visualise the forest depth structure.

**Expected output:** CSV ~215k rows × 11 columns; pickle ~60 MB; GEXF ~200 MB.

In [ ]:
df_comm = pd.DataFrame(
    [{"cell_id": n, "community": community_map[n]} for n in community_map]
)
df_final = df_centrality.merge(
    df_comm[["cell_id", "community"]], on="cell_id", how="left"
)
out_path = RESULTS_DIR / "data" / "italy_etas_network_metrics.csv"
df_final.to_csv(out_path, index=False)
print(f"Results saved → {out_path}  ({len(df_final):,} rows)")

pkl_path = RESULTS_DIR / "cache" / "italy_etas.pkl"
with open(pkl_path, "wb") as f:
    pickle.dump(G, f)
print(f"Network cached → {pkl_path}")

gexf_path = RESULTS_DIR / "gephi" / "italy_etas.gexf"
nx.write_gexf(G, gexf_path)
print(f"Gephi export → {gexf_path}")